In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
ratings = pd.read_csv("../data/ratings.csv")
movies = pd.read_csv("../data/movies.csv")

In [4]:
user_item = ratings.pivot(
    index="userId",
    columns="movieId",
    values="rating"
)
#user-item matrix

In [5]:
user_item_filled = user_item.fillna(0)
'''Fill missing values with 0
Why?
Cosine similarity cannot handle NaN.'''

'Fill missing values with 0\nWhy?\nCosine similarity cannot handle NaN.'

compute user-user similarity

In [6]:
user_similarity = cosine_similarity(user_item_filled)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item.index,
    columns=user_item.index
)

/Users/sabarivishnu/Movierecommendation/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/sabarivishnu/Movierecommendation/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/sabarivishnu/Movierecommendation/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [7]:
user_similarity_df.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
userId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.027283,0.059720,0.194395,0.129080,0.128152,0.158744,0.136968,0.064263,0.016875,...,0.080554,0.164455,0.221486,0.070669,0.153625,0.164191,0.269389,0.291097,0.093572,0.145321
2,0.027283,1.000000,0.000000,0.003726,0.016614,0.025333,0.027585,0.027257,0.000000,0.067445,...,0.202671,0.016866,0.011997,0.000000,0.000000,0.028429,0.012948,0.046211,0.027565,0.102427
3,0.059720,0.000000,1.000000,0.002251,0.005020,0.003936,0.000000,0.004941,0.000000,0.000000,...,0.005048,0.004892,0.024992,0.000000,0.010694,0.012993,0.019247,0.021128,0.000000,0.032119
4,0.194395,0.003726,0.002251,1.000000,0.128659,0.088491,0.115120,0.062969,0.011361,0.031163,...,0.085938,0.128273,0.307973,0.052985,0.084584,0.200395,0.131746,0.149858,0.032198,0.107683
5,0.129080,0.016614,0.005020,0.128659,1.000000,0.300349,0.108342,0.429075,0.000000,0.030611,...,0.068048,0.418747,0.110148,0.258773,0.148758,0.106435,0.152866,0.135535,0.261232,0.060792


top k similar users

In [8]:
def get_top_k_similar_users(target_user_id, k=5):
    similarities = user_similarity_df[target_user_id]
    similarities = similarities.drop(target_user_id)  # remove self
    return similarities.sort_values(ascending=False).head(k)
get_top_k_similar_users(target_user_id=1, k=5)

userId
266    0.357408
313    0.351562
368    0.345127
57     0.345034
91     0.334727
Name: 1, dtype: float64

Recommend unseen movies

In [9]:
def recommend_movies_user_based(target_user_id, k_users=5, n_recommendations=5):
    
    # Movies already rated by target user
    target_user_ratings = user_item.loc[target_user_id]
    watched_movies = target_user_ratings[target_user_ratings.notna()].index

    # Find similar users
    similar_users = get_top_k_similar_users(target_user_id, k_users).index

    # Get ratings from similar users
    similar_users_ratings = user_item.loc[similar_users]

    # Average ratings for each movie
    mean_ratings = similar_users_ratings.mean()

    # Remove movies already watched
    recommendations = mean_ratings.drop(watched_movies)

    # Top-N recommendations
    top_movies = recommendations.sort_values(ascending=False).head(n_recommendations)

    return top_movies


In [10]:
recommend_movies_user_based(target_user_id=1, k_users=5, n_recommendations=5)

movieId
2324    5.0
2467    5.0
2921    5.0
5010    5.0
5026    5.0
dtype: float64

In [11]:
def get_movie_titles(movie_ids):
    return movies[movies["movieId"].isin(movie_ids)][["movieId", "title"]]

get_movie_titles(recommend_movies_user_based(1).index)
#showing movie titles

,movieId,title
1730,2324,Life Is Beautiful (La Vita è bella) (1997)
1855,2467,"Name of the Rose, The (Name der Rose, Der) (1986)"
2197,2921,High Plains Drifter (1973)
3646,5010,Black Hawk Down (2001)
3656,5026,"Brotherhood of the Wolf (Pacte des loups, Le) ..."


In [12]:
def predict_user_cf(user_id, movie_id, k_users=5):
    
    if movie_id not in user_item.columns:
        return np.nan
    
    similar_users = get_top_k_similar_users(user_id, k_users)

    ratings = []
    weights = []

    for sim_user, similarity in similar_users.items():
        rating = user_item.loc[sim_user, movie_id]
        
        if not np.isnan(rating):
            ratings.append(rating * similarity)
            weights.append(similarity)

    if len(weights) == 0:
        return np.nan

    return sum(ratings) / sum(weights)


In [13]:
from sklearn.metrics import mean_squared_error
import numpy as np

actual = []
predicted = []

for row in ratings.sample(500).itertuples():
    pred = predict_user_cf(row.userId, row.movieId)

    if not np.isnan(pred):
        actual.append(row.rating)
        predicted.append(pred)

rmse_user = np.sqrt(mean_squared_error(actual, predicted))
rmse_user


np.float64(1.096304591534486)

In [18]:
import json

results = {
    "rmse_user": float(rmse_user)
}

with open("../outputs/rmse_results.json", "w") as f:
    json.dump(results, f)
